# BC/1.0 多人幾A幾B：控制台（MAIN.ipynb）

這本筆記本把常用操作分成一個個區段，**由上往下按 Shift+Enter 執行**即可：

| 區段 | 做什麼 |
|---|---|
| 0 | 載入共用設定與工具函式（**每次開啟筆記本都要先執行**） |
| 1 | 環境檢查：Python 版本、檔案、埠號、LLM 伺服器是否連得上 |
| 2 | LLM 設定：查看與修改 `bc_game/llm_config.json`、試問 LLM 一次 |
| 3 | 開一局：啟動伺服器、加入 AI、看輸出與結果、全部停止 |
| 4 | 批次對戰：LLM AI 對演算法 AI 連打 N 局並統計 |
| 5 | 測試程式：`bc_llm_test.py`、`bc_test.py`、`bc_selftest.py` |

- 伺服器與 AI 都在**背景**執行，輸出存到 `run_logs/名稱.txt`，用 `tail("名稱")` 查看；每局的協定日誌在 `run_logs/` 底下的子資料夾。
- 預設埠號 **6000**（Windows 上 5040 常被系統服務占用）。
- Windows 與 WSL（Linux）都能用；在 WSL 裡要連 Windows 上的 LLM（Ollama、Lemonade 等），請看「1. 環境檢查」的提示。
- 這本筆記本要放在專案根目錄（與 `bc_game/` 資料夾同一層）。

## 0. 共用設定與工具函式

In [ ]:
import json, os, sys, time, socket, subprocess, urllib.request, importlib
from pathlib import Path

ROOT = Path.cwd()                      # MAIN.ipynb 所在資料夾
if not (ROOT / "bc_game" / "bc.py").exists() and (ROOT.parent / "bc_game" / "bc.py").exists():
    ROOT = ROOT.parent
GAME = ROOT / "bc_game"
BC = GAME / "bc.py"
CONFIG = GAME / "llm_config.json"
PY = sys.executable                    # 用跟這本筆記本相同的 Python
HOST, PORT = "127.0.0.1", 6000         # ← 想換埠號改這裡
RUN = ROOT / "run_logs"
RUN.mkdir(exist_ok=True)
ENV = dict(os.environ, PYTHONIOENCODING="utf-8", PYTHONUTF8="1")   # 子行程輸出一律 UTF-8
PROCS = globals().get("PROCS", {})     # 名稱 -> (Popen, 輸出檔)；重跑這格不會遺失已啟動的行程

assert BC.exists(), f"找不到 {BC}，請把 MAIN.ipynb 放在專案根目錄（bc_game 的上一層）"


def start(name, *args):
    """在背景執行 python bc.py <args>，輸出寫到 run_logs/<name>.txt。"""
    if name in PROCS and PROCS[name][0].poll() is None:
        print(f"{name} 已經在執行（pid {PROCS[name][0].pid}）")
        return
    out = open(RUN / f"{name}.txt", "w", encoding="utf-8")
    p = subprocess.Popen([PY, "-u", str(BC), *map(str, args)], cwd=GAME, env=ENV,
                         stdin=subprocess.DEVNULL, stdout=out, stderr=subprocess.STDOUT)
    PROCS[name] = (p, out)
    print(f"已啟動 {name}（pid {p.pid}）：bc.py {' '.join(map(str, args))}")


def status():
    """列出所有背景行程是否還在執行。"""
    if not PROCS:
        print("目前沒有背景行程")
    for name, (p, _) in PROCS.items():
        code = p.poll()
        print(f"  {name:<12} {'執行中' if code is None else f'已結束（代碼 {code}）'}")


def tail(name, n=20):
    """顯示 run_logs/<name>.txt 的最後 n 行。"""
    f = RUN / f"{name}.txt"
    if not f.exists():
        print(f"沒有 {f.name}")
        return
    lines = f.read_text(encoding="utf-8", errors="replace").splitlines()
    print(f"----- {name}（共 {len(lines)} 行，顯示最後 {min(n, len(lines))} 行）-----")
    print("\n".join(lines[-n:]))


def stop(name):
    p, out = PROCS.pop(name, (None, None))
    if p is None:
        print(f"沒有名為 {name} 的行程")
        return
    if p.poll() is None:
        p.terminate()
        try:
            p.wait(3)
        except subprocess.TimeoutExpired:
            p.kill()
    out.close()
    print(f"已停止 {name}")


def stop_all():
    for name in list(PROCS):
        stop(name)


def wait_all(names, timeout=600):
    """等到指定的行程都結束（最多 timeout 秒），回傳是否全部結束。"""
    t0 = time.time()
    while time.time() - t0 < timeout:
        if all(PROCS[n][0].poll() is not None for n in names if n in PROCS):
            return True
        time.sleep(1)
    return False


def port_free(port):
    with socket.socket() as s:
        try:
            s.bind((HOST, port))
            return True
        except OSError:
            return False


def summarize(log_dir):
    """讀取伺服器日誌，列出每局的結束原因、回合數與名次（以玩家名稱表示）。"""
    rows = []
    for f in sorted(Path(log_dir).glob("game*.jsonl")):
        names, over = {}, None
        for line in f.read_text(encoding="utf-8").splitlines():
            m = json.loads(line).get("msg") or {}
            if m.get("type") == "LOBBY":
                names = {p["id"]: p["name"] for p in m.get("players", [])}
            elif m.get("type") == "GAME_OVER":
                over = m
        if over:
            rank = {names.get(r["player"], r["player"]): r["rank"] for r in over.get("ranking", [])}
            rows.append({"game": f.stem, "reason": over.get("reason"), "turns": over.get("turns"), "rank": rank})
    return rows


ALL_NUMBERS = ["".join(p) for p in __import__("itertools").permutations("0123456789", 4)]


def score(g, s):
    a = sum(x == y for x, y in zip(g, s))
    return a, len(set(g) & set(s)) - a


def _ai_notes(proc_name):
    """從 AI 的輸出檔讀出：每局的秘密、每回合喊數的來源（LLM 理由或退回演算法的原因）。"""
    f = RUN / f"{proc_name}.txt"
    secrets_, notes, game, pending = [], {}, 0, ""
    if not f.exists():
        return secrets_, notes
    for line in f.read_text(encoding="utf-8", errors="replace").splitlines():
        line = line.strip()
        if line.startswith("開局，我的秘密"):
            game += 1
            secrets_.append(line.split()[-1])
        elif line.startswith("LLM 選"):
            pending = "LLM：" + line.split("：", 1)[-1]
        elif line.startswith("LLM 無可用答案") or line.startswith("拒用 LLM"):
            pending = "演算法（" + line.split("（", 1)[-1].split("）")[0] + "）"
        elif line.startswith("第 ") and "回合喊" in line:
            turn = int(line.split()[1])
            notes[(game, turn)] = pending or "演算法"
            pending = ""
    return secrets_, notes


def watch(log_dir, ai_procs=None, until=(), timeout=3600):
    """即時觀戰：持續讀取伺服器日誌，顯示每一手的喊數、各家回報、剩幾種可能，以及 AI 喊數的來源。
    ai_procs：{玩家名稱: 背景行程名稱}，用來讀 AI 的秘密（上帝視角）與 LLM 理由。
    until：這些背景行程都結束後停止觀戰。按筆記本的 ■（中斷）只會停止觀戰，對戰照常進行。"""
    ai_procs = ai_procs or {}
    offsets, seen, t0 = {}, set(), time.time()
    names, cands, game_no, shown_secret = {}, {}, 0, False
    try:
        while time.time() - t0 < timeout:
            new = False
            for f in sorted(Path(log_dir).glob("game*.jsonl")):
                with open(f, encoding="utf-8") as fh:
                    fh.seek(offsets.get(f, 0))
                    chunk = fh.read()
                    end = chunk.rfind("\n") + 1            # 只處理完整的行
                    offsets[f] = offsets.get(f, 0) + len(chunk[:end].encode("utf-8"))
                for line in chunk[:end].splitlines():
                    r = json.loads(line)
                    m = r.get("msg") or {}
                    t = m.get("type")
                    if r.get("dir") != "send" or t not in ("GAME_START", "TURN_RESULT", "REVEALED", "GAME_OVER"):
                        continue
                    key = (f.name, t, m.get("turn_id"), m.get("player"))
                    if key in seen:                       # 廣播會對每位收件者各記一筆，只顯示一次
                        continue
                    seen.add(key)
                    new = True
                    if t == "GAME_START":
                        game_no += 1
                        shown_secret = False
                        names = {p["id"]: p["name"] for p in m.get("players", [])}
                        cands = {pid: ALL_NUMBERS for pid in names}
                        order = " → ".join(names.get(p, p) for p in m.get("order", []))
                        print(f"\n========== 第 {game_no} 局（{f.stem}）順序：{order} ==========")
                    elif t == "TURN_RESULT":
                        if not shown_secret:
                            sec = []
                            for pid, nm in names.items():
                                if nm in ai_procs:
                                    ss, _ = _ai_notes(ai_procs[nm])
                                    if len(ss) >= game_no:
                                        sec.append(f"{nm}={ss[game_no - 1]}")
                            if sec:
                                print("  秘密（上帝視角）：" + "、".join(sec))
                            shown_secret = True
                        g, guess = m.get("guesser"), m.get("guess")
                        who = names.get(g, g)
                        if m.get("skipped") or not guess:
                            print(f"第 {m.get('turn_id'):>2} 回合 {who} 逾時跳過")
                            continue
                        note = ""
                        if who in ai_procs:
                            note = _ai_notes(ai_procs[who])[1].get((game_no, m.get("turn_id")), "")
                        parts = []
                        for res in m.get("results", []):
                            pid, a, b = res.get("player"), res.get("a"), res.get("b")
                            if isinstance(a, int) and isinstance(b, int) and pid in cands:
                                cands[pid] = [s for s in cands[pid] if score(guess, s) == (a, b)]
                                parts.append(f"{names.get(pid, pid)} {a}A{b}B（剩 {len(cands[pid])} 種）")
                        print(f"第 {m.get('turn_id'):>2} 回合 {who} 喊 {guess}：" + "、".join(parts)
                              + (f"   ← {note}" if note else ""))
                        for pid in m.get("eliminated", []):
                            print(f"    ★ {names.get(pid, pid)} 被猜中出局！")
                        for pid in m.get("forfeited", []) + m.get("disqualified", []):
                            print(f"    ✗ {names.get(pid, pid)} 棄權或被取消資格")
                    elif t == "REVEALED":
                        print(f"    {names.get(m.get('player'), m.get('player'))} 揭示秘密 {m.get('secret')}"
                              f"（驗證 {m.get('verdict')}）")
                    elif t == "GAME_OVER":
                        rank = "、".join(f"第 {x.get('rank')} 名 {names.get(x['player'], x['player'])}"
                                        for x in m.get("ranking", []))
                        print(f"--- 結束：{m.get('reason')}，{m.get('turns')} 回合；{rank}")
            if not new and until and all(PROCS[n][0].poll() is not None for n in until if n in PROCS):
                print("\n（對戰都結束了，停止觀戰）")
                return True
            time.sleep(0.5)
    except KeyboardInterrupt:
        print("\n（已停止觀戰；對戰仍在背景進行，可再執行一次繼續觀看，或用 stop_all() 停止）")
        return False
    print("\n（觀戰逾時）")
    return False


def show_duel(log_dir, done):
    """停止批次對戰的行程並統計勝場。"""
    for n in ("duel_llm", "duel_algo", "duel_server"):
        if n in PROCS:
            stop(n)
    rows, wins = summarize(log_dir), {}
    for r in rows:
        winner = min(r["rank"], key=lambda k: r["rank"][k] if r["rank"][k] is not None else 99)
        wins[winner] = wins.get(winner, 0) + 1
        print(f"{r['game']}：{r['turns']} 回合，勝者 {winner}")
    print(("全部打完" if done else "逾時，只統計已完成的局"), f"：共 {len(rows)} 局，勝場 {wins}")


def run(*cmd, timeout=1800, llm=False):
    """在前景執行一個指令（例如測試程式），輸出即時顯示。
    llm=False 時設定 BC_LLM_PROVIDER=off：測試中啟動的 AI／BOT 只用演算法，
    不會去呼叫 llm_config.json 裡的 LLM（否則上百局的純 AI 對局每一手都等 LLM，會跑非常久）。"""
    env = dict(ENV) if llm else dict(ENV, BC_LLM_PROVIDER="off")
    print("執行：", " ".join(map(str, cmd)), "" if llm else "（測試中的 AI 只用演算法）", flush=True)
    p = subprocess.Popen(list(map(str, cmd)), cwd=ROOT, env=env, stdin=subprocess.DEVNULL,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, encoding="utf-8", errors="replace")
    import threading
    fired = []
    killer = threading.Timer(timeout, lambda: (fired.append(1), p.kill()))   # 超過 timeout 秒就強制結束
    killer.start()
    try:
        for line in p.stdout:
            print(line, end="", flush=True)
        p.wait()
    except KeyboardInterrupt:
        p.kill()
        print("\n（已中斷）")
    finally:
        killer.cancel()
    print(f"（逾時 {timeout} 秒，已強制結束）" if fired else f"（結束代碼 {p.returncode}）")


print("專案資料夾：", ROOT)
print("Python：", PY)
print("埠號：", PORT)

## 1. 環境檢查

In [ ]:
print("Python 版本：", sys.version.split()[0], "OK" if sys.version_info >= (3, 10) else "（需要 3.10 以上）")
for f in (BC, GAME / "bc_test.py", GAME / "bc_llm_test.py", ROOT / "bc_selftest.py", CONFIG):
    print(("  有  " if f.exists() else "  缺少") + f"  {f.relative_to(ROOT)}")
print(f"埠號 {PORT}：", "可用" if port_free(PORT) else "已被占用（可能伺服器已在執行，或請改 PORT）")

# LLM 伺服器是否連得上（OpenAI 相容 API 的 GET /models）
import platform, re as _re
from urllib.parse import urlsplit


def probe(base_url, key_env=""):
    req = urllib.request.Request(base_url.rstrip("/") + "/models")
    if key_env and os.environ.get(key_env):
        req.add_header("Authorization", "Bearer " + os.environ[key_env])
    with urllib.request.urlopen(req, timeout=3) as r:
        return [m.get("id") for m in json.load(r).get("data", [])]


IN_WSL = "microsoft" in platform.release().lower()
print("執行環境：", "WSL（Linux）" if IN_WSL else platform.system())
cfg = json.loads(CONFIG.read_text(encoding="utf-8-sig"))
prof = {k: v for k, v in cfg.items() if k != "profiles"} | cfg["profiles"].get(cfg.get("active"), {})
base = prof.get("base_url", "")
try:
    models = probe(base, prof.get("api_key_env"))
    print(f"LLM 伺服器 {base}：連得上，可用模型 {models}")
    if prof.get("model") not in models:
        print(f"  注意：設定的模型 {prof.get('model')} 不在清單中（Ollama 請先 ollama pull {prof.get('model')}）")
except Exception as e:
    print(f"LLM 伺服器 {base}：連不上（{e}）。AI 會自動改用演算法。")
    host = urlsplit(base).hostname
    if IN_WSL and host in ("localhost", "127.0.0.1"):
        # WSL 預設（NAT 網路）連不到 Windows 的 localhost，要改用 Windows 主機的 IP
        try:
            route = subprocess.run(["ip", "route", "show", "default"], capture_output=True, text=True).stdout
            win_ip = _re.search(r"via (\S+)", route).group(1)
        except Exception:
            win_ip = None
        if win_ip:
            alt = base.replace(host, win_ip, 1)
            try:
                probe(alt, prof.get("api_key_env"))
                print(f"  ✔ 改用 Windows 主機 IP 連得上：{alt}")
                print(f'    在 2-2 加一行 cfg["profiles"]["{cfg.get("active")}"]["base_url"] = "{alt}" 後執行即可')
            except Exception:
                print(f"  WSL 連 Windows 上的 LLM 有兩種做法：")
                print(f"   (a) 讓 LLM 伺服器監聽 0.0.0.0（不只 127.0.0.1），base_url 改成 {alt}")
                print(f"       （Windows 防火牆可能要允許該埠）")
                print(f"   (b) 在 Windows 的 %UserProfile%\\.wslconfig 加上 [wsl2] networkingMode=mirrored，")
                print(f"       執行 wsl --shutdown 重開後，WSL 就能直接用 localhost 連到 Windows")

## 2. LLM 設定（`bc_game/llm_config.json`）

AI 啟動時才讀取設定檔，所以**改完後新啟動的 AI 就會套用**（已在執行的 AI 不受影響）。優先順序：命令列參數 > 環境變數 > 設定檔 > 程式預設值。

In [ ]:
# 2-1 查看目前設定
cfg = json.loads(CONFIG.read_text(encoding="utf-8-sig"))
print("目前使用的 profile：", cfg.get("active"), "｜ enabled =", cfg.get("enabled", True),
      "｜ timeout =", cfg.get("timeout"), "｜ temperature =", cfg.get("temperature"))
for name, p in cfg.get("profiles", {}).items():
    mark = "→" if name == cfg.get("active") else " "
    print(f" {mark} {name:<10} {p.get('base_url'):<32} 模型 {p.get('model'):<28} 金鑰變數 {p.get('api_key_env') or '（不需要）'}")

In [ ]:
# 2-2 修改設定：改下面的值後執行這一格就會存檔
cfg = json.loads(CONFIG.read_text(encoding="utf-8-sig"))

cfg["active"] = "ollama"          # 切換提供者：ollama / lmstudio / vllm / openai / 你新增的名稱
cfg["enabled"] = True             # False ＝ AI 只用演算法
cfg["timeout"] = 20               # 單次 LLM 呼叫最多等幾秒
cfg["temperature"] = 0.2

# 換模型（取消註解並修改）：
# cfg["profiles"]["ollama"]["model"] = "llama3.1:8b"

# 新增一個提供者（取消註解並修改）：
# cfg["profiles"]["my_server"] = {"base_url": "http://192.168.0.10:11434/v1", "model": "qwen2.5:7b", "api_key_env": ""}

assert cfg["active"] in cfg["profiles"], f"profiles 裡沒有 {cfg['active']}"
CONFIG.write_text(json.dumps(cfg, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("已儲存", CONFIG.name, "→ active =", cfg["active"], "，模型", cfg["profiles"][cfg["active"]]["model"])

In [ ]:
# 2-3 試問 LLM 一次：模擬一個局面，看 LLM 選了什麼、花多久（連不上會顯示退回演算法）
sys.path.insert(0, str(GAME))
import bc
importlib.reload(bc)                    # bc.py 有修改時重新載入

ai = bc.AI(bc.parse_args(["ai"]))       # 依目前的 llm_config.json 建立 AI（第一行會印出實際設定）
ai.me, ai.guess_timeout = "p1", 60.0
ai.cands = {"p2": list(bc.ALL_NUMBERS), "p3": list(bc.ALL_NUMBERS)}
ai.hist = {"p2": [], "p3": []}
for g in ("0123", "4567"):              # 假設對手的秘密是 5079 與 9812，已經被喊過兩手
    for p, secret in (("p2", "5079"), ("p3", "9812")):
        a, b = bc.score(g, secret)
        ai.cands[p] = [s for s in ai.cands[p] if bc.score(g, s) == (a, b)]
        ai.hist[p].append((g, a, b))

t0 = time.time()
guess = ai.choose(["p1", "p2", "p3"])
print(f"\n喊 {guess}，花 {time.time() - t0:.1f} 秒；來源統計 {ai.stats}")
print("（llm＝採用 LLM 的答案；fallback:原因＝改用演算法）")

## 3. 開一局

依序執行：3-1 啟動伺服器 → 3-2 加入 AI →（3-3 自己也加入）→ **3-4 即時觀戰** → 3-5 看進度 → 3-6 看結果 → 3-7 全部停止。
人數到齊就會自動開局。LLM 每手要數秒，所以喊數時限設長一點（預設 60 秒）。

In [ ]:
# 3-1 啟動伺服器
PLAYERS = 2            # 每局人數（2–6，含 AI）；3-2 加入 2 個 AI，自己也要加入（3-3）就改成 3
BOTS = 0               # 伺服器自動啟動的 AI 數量（BOT 也會讀 llm_config.json）
GAMES = 1              # 連續幾局
GUESS_TIMEOUT = 60     # 喊數時限（秒）

GAME_LOG = RUN / ("game_" + time.strftime("%Y%m%d_%H%M%S"))
start("server", "server", "--port", PORT, "--players", PLAYERS, "--bots", BOTS, "--games", GAMES,
      "--guess-timeout", GUESS_TIMEOUT, "--log-dir", GAME_LOG)
time.sleep(1)
tail("server", 5)

In [ ]:
# 3-2 加入 AI 玩家（名稱不可重複；不需要的那行註解掉）
start("llm_ai",  "ai", "--port", PORT, "--name", "LLM-AI",  "--games", GAMES)                 # 依 llm_config.json 用 LLM
start("algo_ai", "ai", "--port", PORT, "--name", "ALGO-AI", "--games", GAMES, "--llm", "off")  # 只用演算法
# start("llm_ai2", "ai", "--port", PORT, "--name", "LMS-AI", "--games", GAMES, "--llm-profile", "lmstudio")

In [ ]:
# 3-3 自己加入（人類客戶端要用鍵盤輸入，筆記本無法互動）：把印出的那一行貼到終端機執行
# 注意：要在 AI 到齊開局「之前」加入，所以 3-1 的 PLAYERS 要算進自己（例如 2 個 AI ＋ 自己＝3）
NAME = "我"
if os.name == "nt":   # Windows PowerShell：引號包住的路徑要用 & 執行
    print(f'& "{PY}" "{BC}" client --port {PORT} --name {NAME} --team G07')
else:
    print(f'"{PY}" "{BC}" client --port {PORT} --name {NAME} --team G07')

In [ ]:
# 3-4 即時觀戰：每一手的喊數、各家回報、還剩幾種可能、AI 秘密（上帝視角），以及 LLM 的理由或退回演算法的原因
# 按上方的 ■（中斷）只會停止觀戰，對局照常進行；再執行這格可以繼續看
watch(GAME_LOG, {"LLM-AI": "llm_ai", "ALGO-AI": "algo_ai"}, until=["server"])

In [ ]:
# 3-5 看進度（可重複執行）
status()
tail("server", 15)
tail("llm_ai", 10)

In [ ]:
# 3-6 看結果（從伺服器日誌讀出每局名次）
for r in summarize(GAME_LOG):
    print(f"{r['game']}：{r['reason']}，{r['turns']} 回合，名次 {r['rank']}")
for name in ("llm_ai", "algo_ai"):
    f = RUN / f"{name}.txt"
    if f.exists():
        stats = [l for l in f.read_text(encoding="utf-8").splitlines() if "喊數來源統計" in l]
        print(name, stats[-1] if stats else "（沒有 LLM 統計：只用演算法，或這局還沒結束）")

In [ ]:
# 3-7 全部停止
stop_all()

## 4. 批次對戰：LLM AI 對演算法 AI

2 人局連打 N 局，統計勝場。`WATCH = True` 時會**邊打邊顯示每一手**（同 3-4 的即時觀戰）；按 ■ 只停止觀看，對戰繼續，之後執行 4-2 可以接著看。

In [ ]:
# 4-1 批次對戰
N = 5                  # 局數
GUESS_TIMEOUT = 60
WATCH = True           # True＝即時觀戰；False＝安靜等到打完

DUEL_LOG = RUN / ("duel_" + time.strftime("%Y%m%d_%H%M%S"))
start("duel_server", "server", "--port", PORT, "--players", 2, "--games", N,
      "--guess-timeout", GUESS_TIMEOUT, "--log-dir", DUEL_LOG)
time.sleep(1)
start("duel_llm",  "ai", "--port", PORT, "--name", "LLM-AI",  "--games", N)
start("duel_algo", "ai", "--port", PORT, "--name", "ALGO-AI", "--games", N, "--llm", "off")

if WATCH:
    done = watch(DUEL_LOG, {"LLM-AI": "duel_llm", "ALGO-AI": "duel_algo"}, until=["duel_llm", "duel_algo"],
                 timeout=N * 600)
    if not done:
        raise SystemExit("已停止觀戰，對戰仍在背景進行：執行下一格可繼續觀看並在結束後統計")
else:
    done = wait_all(["duel_llm", "duel_algo"], timeout=N * 600)
show_duel(DUEL_LOG, done)

In [ ]:
# 4-2 中途停止觀戰後，用這格繼續觀看，打完後統計勝場
done = watch(DUEL_LOG, {"LLM-AI": "duel_llm", "ALGO-AI": "duel_algo"}, until=["duel_llm", "duel_algo"],
             timeout=N * 600)
show_duel(DUEL_LOG, done)

## 5. 測試程式

測試中啟動的 AI／BOT 一律**只用演算法**（`run()` 會設定 `BC_LLM_PROVIDER=off`），因為測試檢查的是通訊協定，而且上百局對局若每手都等 LLM 會跑非常久。輸出會即時顯示。

In [ ]:
# 5-1 LLM 模式單元測試（假的 OpenAI 相容伺服器，約 10 秒）
run(PY, GAME / "bc_llm_test.py")

In [ ]:
# 5-2 自寫黑箱測試 bc_test.py：伺服器、AI、人類客戶端共 72 項（約 20 秒）
q = lambda p: f'"{p}"'
run(PY, GAME / "bc_test.py", "--port", 6100,
    "--server", f"{q(PY)} {q(BC)} server", "--ai", f"{q(PY)} {q(BC)} ai", "--client", f"{q(PY)} {q(BC)} client")

In [ ]:
# 5-3 課程提供的 bc_selftest.py：伺服器 48 項（含 2–6 人純 AI 各 10 局，約 3–5 分鐘；SKIP_BOTS=True 可略過）
SKIP_BOTS = False
q = lambda p: f'"{p}"'
run(PY, ROOT / "bc_selftest.py", "--port", 6300, "--server", f"{q(PY)} {q(BC)} server",
    *(["--skip-bots"] if SKIP_BOTS else []))